# Swiggy Dataset Cleaning & Preprocessing Pipeline

**Project:** Swiggy Data Analytics Project  
**Objective:** Transform raw, noisy Swiggy restaurant data into a fully cleaned, validated, and structured format ready for Exploratory Data Analysis (EDA), dashboarding (Power BI / Tableau), and machine learning.

### Cleaning Steps Covered:
1. **Data Ingestion & Diagnostics** (Examining data types, missing values, duplicates)
2. **Deduplication** (Removing exact duplicate records)
3. **Column Standardization** (Snake_case naming conventions)
4. **Missing Value Imputation** (Handling `Area`, `Cuisine`, and `Offer Name`)
5. **Text Sanitization & Feature Engineering** (`primary_cuisine`, `cuisine_count`)
6. **Metric Parsing** (Converting `Rating` and `Number of Ratings` to numeric types)
7. **Price Parsing & Outlier Correction** (Extracting integer `cost_for_two`)
8. **Boolean Mapping & Offer Formatting** (`is_pure_veg`, removing line breaks)
9. **Data Validation & Export** (`swiggy_cleaned.csv`)

## Step 1: Import Libraries and Load Raw Data

In [1]:
import re
import numpy as np
import pandas as pd

# Load raw Swiggy dataset
raw_file_path = 'swiggy_file.csv/swiggy_file.csv'
df_raw = pd.read_csv(raw_file_path)
print(f"Raw dataset shape: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns")
df_raw.head()

Raw dataset shape: 140,657 rows, 10 columns


## Step 2: Initial Diagnostics (Missing Values & Duplicates)

In [2]:
# Check data types and null values
null_summary = pd.DataFrame({
    'Data Type': df_raw.dtypes,
    'Null Count': df_raw.isnull().sum(),
    'Null Percentage (%)': (df_raw.isnull().sum() / len(df_raw) * 100).round(2)
})
print("Missing Value Summary:")
display(null_summary)

exact_duplicates = df_raw.duplicated().sum()
print(f"Exact Duplicate Rows: {exact_duplicates:,}")

Missing Value Summary:
                  Data Type  Null Count  Null Percentage (%)
Restaurant Name         str           0                 0.00
Cuisine                 str          27                 0.02
Rating                  str           0                 0.00
Number of Ratings       str       14542                10.34
Average Price           str           0                 0.00
Number of Offers      int64           0                 0.00
Offer Name              str        1808                 1.29
Area                    str           2                 0.00
Pure Veg                str           0                 0.00
Location                str           0                 0.00
Exact Duplicate Rows: 1,336


## Step 3: Remove Exact Duplicates

In [3]:
# Drop exact duplicates across all columns
df = df_raw.drop_duplicates().copy().reset_index(drop=True)
print(f"Rows before: {len(df_raw):,} | Rows after deduplication: {len(df):,}")

Rows before: 140,657 | Rows after deduplication: 139,321


## Step 4: Standardize Column Names
Standardizing column names into `snake_case` allows clean attribute access and consistent SQL/Pandas querying.

In [4]:
rename_mapping = {
    'Restaurant Name': 'restaurant_name',
    'Cuisine': 'cuisine',
    'Rating': 'rating_raw',
    'Number of Ratings': 'rating_count_raw',
    'Average Price': 'cost_for_two_raw',
    'Number of Offers': 'offer_count',
    'Offer Name': 'offer_name_raw',
    'Area': 'area',
    'Pure Veg': 'pure_veg_raw',
    'Location': 'location'
}
df = df.rename(columns=rename_mapping)
df.columns

## Step 5: Clean String Columns & Impute Missing Values
- Strip leading and trailing whitespace.
- Impute missing `area` (2 rows in Shillong) with the `location` value.
- Impute missing `cuisine` (27 rows) with `'Unknown'`.

In [5]:
# Strip whitespace on text columns
text_cols = ['restaurant_name', 'location', 'area', 'cuisine']
for col in text_cols:
    df[col] = df[col].astype(str).str.strip().replace({'nan': np.nan})

# Impute missing Area
df['area'] = df['area'].fillna(df['location'])

# Impute missing Cuisine
df['cuisine'] = df['cuisine'].fillna('Unknown')

# Normalize comma-separated cuisine spacing
def normalize_cuisine(val):
    if val == 'Unknown':
        return 'Unknown'
    items = [i.strip() for i in str(val).split(',') if i.strip()]
    return ', '.join(items) if items else 'Unknown'

df['cuisine'] = df['cuisine'].apply(normalize_cuisine)

# Feature Engineering: Primary Cuisine & Cuisine Count
df['primary_cuisine'] = df['cuisine'].apply(lambda x: x.split(',')[0].strip())
df['cuisine_count'] = df['cuisine'].apply(
    lambda x: 0 if x == 'Unknown' else len([i for i in x.split(',') if i.strip()])
)

print("Sample engineered cuisine columns:")
display(df[['cuisine', 'primary_cuisine', 'cuisine_count']].head())

Sample engineered cuisine columns:
                cuisine primary_cuisine  cuisine_count
0        Pizzas, Pastas          Pizzas              2
1  Indian, North Indian          Indian              2
2    Italian, Beverages         Italian              2
3       Pizzas, Burgers          Pizzas              2
4     Fast Food, Snacks       Fast Food              2


## Step 6: Parse `Rating` and `Number of Ratings`
- The raw `Rating` column contains `'--'` (unrated) and `'NEW'` (newly listed restaurants).
- We create a categorical `rating_status` (`Rated`, `New`, `Unrated`) and convert numeric ratings to `float64`.
- The raw `Number of Ratings` column contains strings like `'100+ ratings'`, `'1K+ ratings'`, `'Too Few Ratings'`. We extract a clean numeric `rating_count`.

In [6]:
# 1. Rating Status
def get_rating_status(val):
    if pd.isna(val) or val == '--':
        return 'Unrated'
    elif str(val).strip().upper() == 'NEW':
        return 'New'
    return 'Rated'

df['rating_status'] = df['rating_raw'].apply(get_rating_status)
df['rating'] = pd.to_numeric(
    df['rating_raw'].replace({'--': np.nan, 'NEW': np.nan}),
    errors='coerce'
).round(2)

# 2. Rating Count
def parse_rating_count(val):
    if pd.isna(val) or val == 'Too Few Ratings':
        return np.nan
    s = str(val).lower().replace('ratings', '').replace('rating', '').strip().replace('+', '')
    if 'k' in s:
        try:
            return float(s.replace('k', '').strip()) * 1000
        except ValueError:
            return np.nan
    try:
        return float(s)
    except ValueError:
        return np.nan

df['rating_count'] = df['rating_count_raw'].apply(parse_rating_count)

print("Rating status breakdown:")
print(df['rating_status'].value_counts())
print("\nParsed Rating summary (for rated restaurants):")
print(df['rating'].describe().round(2))

Rating status breakdown:
rating_status
Rated      105864
Unrated     19021
New         14436
Name: count, dtype: int64

Parsed Rating summary (for rated restaurants):
count    105864.00
mean          4.04
std           0.50
min           1.00
25%           3.80
50%           4.10
75%           4.30
max           5.00
Name: rating, dtype: float64


## Step 7: Parse Average Price (`cost_for_two`) and Correct Outliers
- Extract digits from `'₹250 for two'` -> 250.
- Detect and fix data entry typos (e.g. ₹199,246 for two in Lalitpur snack shop).

In [7]:
# Extract numeric price
df['cost_for_two'] = df['cost_for_two_raw'].str.extract(r'(\d+)')[0].astype(int)

# Check for extreme typos (> 50,000)
extreme_outliers = df[df['cost_for_two'] > 50000]
if len(extreme_outliers) > 0:
    print(f"Correcting {len(extreme_outliers)} extreme typo(s):")
    for idx in extreme_outliers.index:
        old_cost = df.loc[idx, 'cost_for_two']
        # Correct ₹199246 to ₹199
        new_cost = int(str(old_cost)[:3])
        df.loc[idx, 'cost_for_two'] = new_cost
        print(f"- Row {idx} ({df.loc[idx, 'restaurant_name']}): Rs. {old_cost} -> Rs. {new_cost}")

print("\nCost for two statistics (INR):")
print(df['cost_for_two'].describe().round(1))

Correcting 1 extreme typo(s):
- Row 81455 (The Food Town): Rs. 199246 -> Rs. 199

Cost for two statistics (INR):
count    139321.0
mean        268.9
std         146.5
min           1.0
25%         200.0
50%         250.0
75%         300.0
max        8000.0
Name: cost_for_two, dtype: float64


## Step 8: Standardize Pure Veg & Offer Name Formatting
- Map `Pure Veg` to boolean `is_pure_veg` (`True`/`False`).
- When `offer_count == 0`, impute `offer_name` with `'No Offer'`.
- Remove embedded multiline `\n` line breaks from `offer_name` and replace with `' | '` so downstream tools do not break.

In [8]:
# 1. Pure Veg
df['is_pure_veg'] = df['pure_veg_raw'].astype(str).str.strip().str.lower().eq('yes')

# 2. Offer Count & Offer Name
df['offer_count'] = df['offer_count'].fillna(0).astype(int)

def clean_offer_text(text):
    if pd.isna(text) or str(text).strip() == '':
        return 'No Offer'
    parts = [part.strip() for part in re.split(r'[\r\n]+', str(text)) if part.strip()]
    return ' | '.join(parts) if parts else 'No Offer'

df['offer_name'] = df['offer_name_raw'].apply(clean_offer_text)

print("Pure Veg value counts:")
print(df['is_pure_veg'].value_counts())
print("\nSample cleaned offers:")
display(df[['offer_count', 'offer_name']].head())

Pure Veg value counts:
is_pure_veg
False    80826
True     58495
Name: count, dtype: int64

Sample cleaned offers:
   offer_count                                                                                                                                                                                                                               offer_name
0            2                                                                                                                                                 FLAT DEAL | FLAT ₹125 OFF | USE FLAT125ABOVE ₹699, FLAT ₹120 OFF | USE AXIS120ABOVE ₹500
1            2                                                                                                                                                           30% OFF UPTO ₹75 | USE TRYNEWABOVE ₹149, FLAT ₹120 OFF | USE AXIS120ABOVE ₹500
2            1                                                                                                                               

## Step 9: Column Selection, Final Validation & Export

In [9]:
final_columns = [
    'restaurant_name',
    'location',
    'area',
    'cuisine',
    'primary_cuisine',
    'cuisine_count',
    'rating',
    'rating_status',
    'rating_count',
    'cost_for_two',
    'is_pure_veg',
    'offer_count',
    'offer_name'
]

df_clean = df[final_columns].copy()

# Export to CSV
output_csv = 'swiggy_cleaned.csv'
df_clean.to_csv(output_csv, index=False, encoding='utf-8')
print(f"Successfully exported cleaned dataset to '{output_csv}'!")

# Validation Summary
print("\nFinal Dataset Info:")
df_clean.info()

Successfully exported cleaned dataset to 'swiggy_cleaned.csv'!

Final Dataset Info:
<class 'pandas.DataFrame'>
RangeIndex: 139321 entries, 0 to 139320
Data columns (total 13 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   restaurant_name  139321 non-null  str    
 1   location         139321 non-null  str    
 2   area             139321 non-null  str    
 3   cuisine          139321 non-null  str    
 4   primary_cuisine  139321 non-null  str    
 5   cuisine_count    139321 non-null  int64  
 6   rating           105864 non-null  float64
 7   rating_status    139321 non-null  str    
 8   rating_count     105864 non-null  float64
 9   cost_for_two     139321 non-null  int64  
 10  is_pure_veg      139321 non-null  bool   
 11  offer_count      139321 non-null  int64  
 12  offer_name       139321 non-null  str    
dtypes: bool(1), float64(2), int64(3), str(7)
memory usage: 12.9 MB


## Step 10: Exploratory Preview & Quick Insights

In [10]:
print("=== TOP 10 CUISINES ON SWIGGY ===")
print(df_clean['primary_cuisine'].value_counts().head(10))

print("\n=== TOP 10 CITIES BY RESTAURANT COUNT ===")
print(df_clean['location'].value_counts().head(10))

print("\n=== AVERAGE COST FOR TWO BY VEG vs NON-VEG ===")
print(df_clean.groupby('is_pure_veg')['cost_for_two'].mean().round(2))

print("\n=== SAMPLE CLEANED RECORDS ===")
display(df_clean.head(5))

=== TOP 10 CUISINES ON SWIGGY ===
primary_cuisine
North Indian    20154
Chinese         18844
Indian          16452
Biryani          9798
Pizzas           8602
South Indian     8190
Bakery           7806
Beverages        7272
Snacks           6287
Fast Food        5411
Name: count, dtype: int64

=== TOP 10 CITIES BY RESTAURANT COUNT ===
location
Kanpur         2000
Central-Goa    1998
Hyderabad      1998
Indore         1998
Nagpur         1998
Vadodara       1998
Lucknow        1996
Jaipur         1990
Kanchrapara    1977
Kolkata        1974
Name: count, dtype: int64

=== AVERAGE COST FOR TWO BY VEG vs NON-VEG ===
is_pure_veg
False    289.82
True     240.03
Name: cost_for_two, dtype: float64

=== SAMPLE CLEANED RECORDS ===
   restaurant_name location                    area               cuisine primary_cuisine  cuisine_count  rating rating_status  rating_count  cost_for_two  is_pure_veg  offer_count                                                                                       